# 02 - Limpieza y Preparación de Datos
## Hotel Dann Monasterio - Proyecto de Analítica Descriptiva

## Objetivo del notebook
Limpiar el dataset consolidado producto del notebook `01_exploracion_dataset.ipynb` y dejarlo listo para el análisis exploratorio y descriptivo. Las tareas son:

1. Cargar el dataset consolidado de los huéspedes (Hoja1 + Hoja2).
2. Eliminar columnas con 100% de valores nulos.
3. Eliminar columnas sin variabilidad (un único valor).
4. Anonimizar datos personales (PII) mediante hash SHA-256.
5. Tratar valores atípicos en `edad_aco` (valores fuera de rango plausible).
6. Estandarizar tipos de fechas.
7. Calcular variables derivadas: `duracion_estancia`, `lead_time`, `anio`, `mes`, `dia_semana`, `ingreso_total`, `rango_edad`.
8. Eliminar duplicados.
9. Persistir el dataset limpio en `data/processed/reservas_clean.parquet`.

## Contexto CRISP-DM
Este notebook corresponde a la fase **Preparación de los datos**. Su salida es el insumo de los notebooks 03 (EDA) y 04 (Análisis Descriptivo).

## Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 80)

## Cargar dataset original

Cargamos las dos hojas del Excel y las consolidamos. El resultado es el mismo `df` que se exploró en el notebook 01.

In [ ]:
ruta = Path("../data/raw/DataSet_ReservaYHuespedes_V01.xlsx")

hoja1 = pd.read_excel(ruta, sheet_name="Hoja1")
hoja2 = pd.read_excel(ruta, sheet_name="Hoja2")
df = pd.concat([hoja1, hoja2], ignore_index=True)

print(f"Shape inicial: {df.shape}")
df.head(3)

## Crear copia para trabajar

Siempre trabajamos sobre una copia, así conservamos el DataFrame original por si necesitamos volver atrás.

In [ ]:
df_clean = df.copy()
print(f"Trabajando sobre copia: {df_clean.shape}")

## Paso 1 - Eliminar columnas con 100% de valores nulos

In [ ]:
nulos_pct = df_clean.isnull().mean() * 100
cols_100_nulas = nulos_pct[nulos_pct == 100].index.tolist()

print(f"Columnas 100% nulas a eliminar: {len(cols_100_nulas)}")
for c in cols_100_nulas:
    print(f"  - {c}")

df_clean = df_clean.drop(columns=cols_100_nulas)
print(f"\nShape después: {df_clean.shape}")

## Paso 2 - Eliminar columnas sin variabilidad (un único valor)

In [ ]:
unicos = df_clean.nunique(dropna=True)
cols_un_valor = unicos[unicos == 1].index.tolist()

print(f"Columnas con un único valor a eliminar: {len(cols_un_valor)}")
for c in cols_un_valor:
    print(f"  - {c}: {df_clean[c].dropna().unique()}")

df_clean = df_clean.drop(columns=cols_un_valor)
print(f"\nShape después: {df_clean.shape}")

## Paso 3 - Anonimización de datos personales (PII)

Las siguientes columnas contienen información personal identificable y deben transformarse antes de usarlas en cualquier análisis:

- `ident_aco` (número de documento) → reemplazar por hash SHA-256 truncado a 16 caracteres.
- `nombre_aco` (nombre completo) → eliminar (no aporta al análisis y es PII directa).

**Fundamento legal**: Ley 1581 de 2012 y Decreto 1377 de 2013 (Colombia).

In [ ]:
def hash_id(x):
    """Genera un hash SHA-256 truncado a 16 caracteres a partir de un identificador."""
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:16]

# 1) Hashear ident_aco
df_clean["id_huesped"] = df_clean["ident_aco"].apply(hash_id)

# 2) Eliminar nombre completo e identificación original
cols_pii = [c for c in ["ident_aco", "nombre_aco"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_pii)

print("Columnas PII eliminadas/transformadas:", cols_pii)
print("\nEjemplo de id_huesped anonimizado:")
df_clean[["id_huesped"]].head(3)

## Paso 4 - Tratamiento de valores atípicos en `edad_aco`

En el notebook 01 detectamos edades imposibles (0 y 126 años). Política: marcamos como inválidas las edades fuera de 1-100 años. **No eliminamos las filas**, sólo marcamos la edad para que no sesgue el análisis demográfico.

In [ ]:
antes = df_clean["edad_aco"].describe()
print("Antes del tratamiento:")
print(antes)

df_clean["edad_valida"] = df_clean["edad_aco"].between(1, 100)
df_clean["edad_aco_limpia"] = np.where(df_clean["edad_valida"], df_clean["edad_aco"], np.nan)

print(f"\nRegistros con edad fuera de rango: {(~df_clean['edad_valida']).sum():,}")
print("\nDescripción después del tratamiento (edad_aco_limpia):")
print(df_clean["edad_aco_limpia"].describe())

## Paso 5 - Estandarizar tipos de fechas

Convertimos todas las columnas de fecha a `datetime64` para facilitar las operaciones temporales.

In [ ]:
cols_fecha = ["fllega_aco", "fsalid_aco", "fcheckout", "fechasischin"]
for c in cols_fecha:
    if c in df_clean.columns:
        df_clean[c] = pd.to_datetime(df_clean[c], errors="coerce")

# fecha viene como 'AAAA.MM.DD' (string)
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], format="%Y.%m.%d", errors="coerce")

df_clean[cols_fecha + ["fecha"]].dtypes

## Paso 6 - Variables derivadas

Calculamos columnas nuevas a partir de las fechas y los ingresos. Estas variables serán usadas intensamente en los notebooks 03 y 04.

In [ ]:
# Duración de la estancia (en noches)
df_clean["duracion_estancia"] = (df_clean["fsalid_aco"] - df_clean["fllega_aco"]).dt.days

# Lead time (días entre la fecha de registro y la llegada)
# Aproximación: usamos 'fecha' (registro) y 'fllega_aco' (llegada).
df_clean["lead_time"] = (df_clean["fllega_aco"] - df_clean["fecha"]).dt.days

# Variables temporales
df_clean["anio"] = df_clean["fllega_aco"].dt.year
df_clean["mes"] = df_clean["fllega_aco"].dt.month
df_clean["trimestre"] = df_clean["fllega_aco"].dt.quarter
df_clean["dia_semana"] = df_clean["fllega_aco"].dt.day_name()

# Periodo COVID
def periodo_covid(anio):
    if anio in (2020, 2021):
        return "Pandemia"
    if anio == 2022:
        return "Recuperación"
    return "Post-pandemia"

df_clean["periodo_covid"] = df_clean["anio"].apply(periodo_covid)

# Ingreso total = plan + adicionales
df_clean["ingreso_total"] = (
    df_clean["totalconsumosplan"].fillna(0) +
    df_clean["totalconsumosadicional"].fillna(0)
)

# Rango de edad para análisis demográfico
bins = [0, 18, 25, 35, 50, 65, 100]
labels = ["<18", "18-25", "26-35", "36-50", "51-65", "66+"]
df_clean["rango_edad"] = pd.cut(df_clean["edad_aco_limpia"], bins=bins, labels=labels)

df_clean[["duracion_estancia", "lead_time", "anio", "mes", "dia_semana", "periodo_covid", "ingreso_total", "rango_edad"]].head()

### Validar la duración de estancia

Esperamos valores positivos. Si hay registros con `fsalid_aco < fllega_aco`, son errores que se marcan.

In [ ]:
neg = (df_clean["duracion_estancia"] < 0).sum()
ceros = (df_clean["duracion_estancia"] == 0).sum()
print(f"Registros con duración negativa: {neg:,}")
print(f"Registros con duración = 0 días: {ceros:,}")
print(f"\nDistribución de duración de estancia:")
print(df_clean["duracion_estancia"].describe())

## Paso 7 - Reemplazar infinitos y manejar nulos

Estandarizamos `inf`/`-inf` a NaN para evitar errores en cálculos posteriores.

In [ ]:
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Tabla resumen final de nulos por columna
resumen = pd.DataFrame({
    "nulos": df_clean.isnull().sum(),
    "% nulos": (df_clean.isnull().mean()*100).round(2)
}).sort_values("% nulos", ascending=False)
resumen.head(15)

## Paso 8 - Eliminar duplicados

Buscamos duplicados exactos (todas las columnas idénticas). Por construcción del PMS no esperamos muchos.

In [ ]:
dup = df_clean.duplicated().sum()
print(f"Duplicados exactos: {dup:,}")
df_clean = df_clean.drop_duplicates()
print(f"Shape final tras drop_duplicates: {df_clean.shape}")

## Paso 9 - Selección final de variables del proyecto

Para los notebooks siguientes nos quedamos con las 36 variables del proyecto + las derivadas calculadas en este notebook.

In [ ]:
VARS_FINAL = [
    # Identificador
    "id_huesped",
    # Tiempo originales
    "fecha", "fllega_aco", "fsalid_aco", "fcheckout", "fechasischin",
    # Tiempo derivadas
    "anio", "mes", "trimestre", "dia_semana", "periodo_covid",
    "duracion_estancia", "lead_time",
    # Segmento y motivación
    "codsegmento", "codmotivacion",
    # Plan tarifario
    "codigp_pla", "descri_pla", "alimen_pla", "tarifa", "adicional",
    # Temporada
    "codigotemporada", "nombretemporada",
    # Canal / agencia
    "nombre_age", "codiga_age", "nombre_emp",
    # Habitación
    "tiphab_tip", "clahab_clh", "nrohab_hab",
    # Huésped
    "edad_aco_limpia", "rango_edad", "sexo_aco",
    "nacionalidad", "clasi_aco", "idn_aco", "incognito",
    # Ingresos
    "valorplan", "ivaplan", "servicioplan",
    "valorconsumoadicional", "totalconsumosplan", "totalconsumosadicional",
    "ingreso_total",
    # Folio / reserva
    "numvoucher", "nreser_res", "folio_titular",
]

# Filtramos sólo columnas existentes
VARS_FINAL = [v for v in VARS_FINAL if v in df_clean.columns]
df_final = df_clean[VARS_FINAL].copy()

print(f"Shape final: {df_final.shape}")
df_final.head(3)

## Paso 10 - Persistir el dataset limpio

Guardamos en formato Parquet (más rápido y más compacto que CSV) en `data/processed/reservas_clean.parquet`.
También guardamos una versión CSV de respaldo.

In [ ]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_parquet = out_dir / "reservas_clean.parquet"
out_csv     = out_dir / "reservas_clean.csv"

try:
    df_final.to_parquet(out_parquet, index=False)
    print(f"Parquet guardado: {out_parquet}")
except Exception as e:
    print(f"No se pudo guardar Parquet ({e}); guardamos sólo CSV.")

df_final.to_csv(out_csv, index=False)
print(f"CSV guardado:     {out_csv}")

print(f"\nRegistros: {len(df_final):,}  |  Columnas: {df_final.shape[1]}")

# Conclusiones del notebook 02

1. Se eliminaron **11 columnas con 100% de valores nulos** y **8 columnas sin variabilidad**, reduciendo el dataset de 79 a aproximadamente 60 columnas operativas.
2. Se **anonimizó** la identificación del huésped con SHA-256 (id_huesped) y se eliminó el nombre completo.
3. Se **marcaron como inválidas** las edades fuera del rango plausible 1-100 años.
4. Se estandarizaron todas las columnas de fecha a `datetime64`.
5. Se calcularon **9 variables derivadas** clave para los análisis posteriores: `duracion_estancia`, `lead_time`, `anio`, `mes`, `trimestre`, `dia_semana`, `periodo_covid`, `ingreso_total`, `rango_edad`.
6. El dataset final tiene aproximadamente **65.000 filas y ~45 columnas analíticas**, almacenado en `data/processed/reservas_clean.parquet`.

**Siguiente paso**: notebook `03_analisis_exploratorio.ipynb`, donde realizamos el análisis exploratorio bivariado, multivariado y de series temporales.